In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from boltz.data.write.writer import BoltzWriter

import mdtraj as md

from boltz.data.types import Structure, StructureV2
from boltz.data.write.mmcif import to_mmcif
from boltz.data.write.pdb import to_pdb
import numpy as np
from dataclasses import asdict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Example: programmatic use of BoltzWriter to convert .npz structures to .cif or .pdb
# Uses BoltzWriter from src/boltz/data/write/writer.py


# Inputs
data_dir = Path("/data2/scratch/group_scratch/boltz_train/rcsb_processed_targets/structures")   # directory containing {record_id}.npz
output_dir = Path("./")                # where .cif/.pdb will be written
output_format = "pdb"                             # mmcif or "pdb"
boltz2 = False                                      # set True if structures are boltz2 format

# Create writer
writer = BoltzWriter(
    data_dir=str(data_dir),
    output_dir=str(output_dir),
    output_format=output_format,
    boltz2=boltz2,
)

# The writer is designed as a PyTorch-Lightning BasePredictionWriter and is normally called
# inside a training/prediction loop. For a simple conversion script, you can mimic the
# minimal pieces the writer needs: a record object with .id and a corresponding .npz file
# in data_dir. Here we show how to call to_pdb / to_mmcif directly using the same logic.


def convert_record(record_id: str, output_format: str = "mmcif"):
    path = data_dir / f"{record_id}.npz"
    # Load as Structure or StructureV2 depending on boltz2 flag
    if boltz2:
        structure = StructureV2.load(path)
    else:
        structure = Structure.load(path)

    # If you have predicted coordinates you would update structure.atoms["coords"]
    # For conversion of existing stored structure, just use to_mmcif / to_pdb
    cif_text = to_mmcif(structure, plddts=None, boltz2=boltz2)
    out_path = output_dir / f"{record_id}.cif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(cif_text)
    print(f"Wrote {out_path}")
    if output_format == "pdb":
        print(out_path)
        protein = md.load(str(out_path))
        protein.save_pdb(out_path.with_suffix(".pdb"))
        print(f"Wrote {out_path.with_suffix('.pdb')}")

# Example usage: convert a list of record ids
record_ids = ["2ags", "2hw5"]
for rid in record_ids:
    convert_record(rid, output_format=output_format)

Wrote 2ags.cif
2ags.cif
Wrote 2ags.pdb
Wrote 2hw5.cif
2hw5.cif
Wrote 2hw5.pdb


In [6]:
protein = md.load("./2ags.cif")
protein.save_pdb("./2ags.pdb")